Claim Extraction with Qwen2.5

In [ ]:
import os
import re
import json
import ast
import pandas as pd
import torch
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM

INPUT_CSV = r"../Results/Phi3_Responses.csv"



MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"

OUTPUT_CSV = f"../Data/Phi3_Responses_with_Claims_by_{MODEL_NAME.split('/')[-1]}.csv"

CLAIM_LEVEL_CSV = f"../Data/Phi3_Claims_Exploded_by_{MODEL_NAME.split('/')[-1]}.csv"

print("Input:", INPUT_CSV)
print("Output:", OUTPUT_CSV)
print("Claim-level output:", CLAIM_LEVEL_CSV)

In [ ]:
df = pd.read_csv(INPUT_CSV)

print("Rows:", len(df))
print("Columns:", df.columns.tolist())

required = {"Question", "Phi3_Answer"}
missing = required - set(df.columns)
if missing:
    raise ValueError(f"Missing required columns: {missing}")

df[["Question_ID", "Question", "Phi3_Answer"]].head()

## Load Phi-3

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype="auto",
    device_map="auto"
)

model.eval()

print("Loaded:", MODEL_NAME)

In [ ]:
SYSTEM_PROMPT = """
You are a factual claim extraction assistant.

Your task is to decompose an answer into atomic, independently verifiable factual claims.

Rules:
1. Each claim must contain exactly ONE factual proposition.
2. Split compound statements into separate claims.
3. Preserve important entities, dates, quantities, locations, and relationships.
4. Resolve simple pronouns when possible so each claim can stand alone.
5. Do NOT include opinions, advice, hedging, greetings, refusals, or non-factual filler.
6. Do NOT judge whether the claims are true or false.
7. Do NOT add new information.
8. If the answer contains no verifiable factual claim, return an empty list.
9. Return valid JSON only, in exactly this format:
   {"claims": ["claim 1", "claim 2"]}
"""

def build_messages(question, answer):
    user_prompt = f"""
Question:
{question}

Generated Answer:
{answer}

Extract the atomic factual claims from the generated answer.
"""
    return [
        {"role": "system", "content": SYSTEM_PROMPT.strip()},
        {"role": "user", "content": user_prompt.strip()}
    ]

In [ ]:
def parse_claims(text):
    """
    Robustly parse Phi-3 output into a Python list of claim strings.
    """
    if not isinstance(text, str):
        return []

    text = text.strip()

    text = re.sub(r"^```(?:json)?\s*", "", text, flags=re.I)
    text = re.sub(r"\s*```$", "", text)

    try:
        obj = json.loads(text)
        if isinstance(obj, dict) and isinstance(obj.get("claims"), list):
            return [str(x).strip() for x in obj["claims"] if str(x).strip()]
    except Exception:
        pass

    match = re.search(r"\{.*\}", text, flags=re.S)
    if match:
        candidate = match.group(0)
        try:
            obj = json.loads(candidate)
            if isinstance(obj, dict) and isinstance(obj.get("claims"), list):
                return [str(x).strip() for x in obj["claims"] if str(x).strip()]
        except Exception:
            pass

    try:
        obj = ast.literal_eval(text)
        if isinstance(obj, dict) and isinstance(obj.get("claims"), list):
            return [str(x).strip() for x in obj["claims"] if str(x).strip()]
    except Exception:
        pass

    claims = []
    for line in text.splitlines():
        line = re.sub(r"^\s*(?:[-*]|\d+[.)])\s*", "", line).strip()
        if line and not line.startswith("{") and not line.startswith("}"):
            claims.append(line)

    return claims

In [ ]:
@torch.inference_mode()
def extract_claims(question, answer, max_new_tokens=300):
    messages = build_messages(question, answer)

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=3500
    ).to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        temperature=0.0,
        pad_token_id=tokenizer.eos_token_id
    )

    generated = outputs[0][inputs["input_ids"].shape[1]:]
    text = tokenizer.decode(generated, skip_special_tokens=True).strip()

    claims = parse_claims(text)

    return {
        "raw_output": text,
        "claims": claims
    }

## Test on one answer

In [ ]:
test_idx = 0

question = df.loc[test_idx, "Question"]
answer = df.loc[test_idx, "Phi3_Answer"]

print("QUESTION:")
print(question)
print("\nPHI-3 ANSWER:")
print(answer)

result = extract_claims(question, answer)

print("\nRAW EXTRACTION:")
print(result["raw_output"])

print("\nPARSED CLAIMS:")
for i, claim in enumerate(result["claims"], 1):
    print(f"{i}. {claim}")

## Run claim extraction over the dataset

In [ ]:

if os.path.exists(OUTPUT_CSV):
    work_df = pd.read_csv(OUTPUT_CSV)
    print("Resuming from existing output.")
else:
    work_df = df.copy()
    work_df["Atomic_Claims"] = ""
    work_df["Claim_Count"] = 0
    work_df["Claim_Extraction_Raw"] = ""

for idx in tqdm(range(len(work_df))):
    existing = work_df.at[idx, "Atomic_Claims"]

    if isinstance(existing, str) and existing.strip().startswith("["):
        continue

    question = str(work_df.at[idx, "Question"])
    answer = str(work_df.at[idx, "Phi3_Answer"])

    try:
        result = extract_claims(question, answer)

        work_df.at[idx, "Atomic_Claims"] = json.dumps(
            result["claims"],
            ensure_ascii=False
        )
        work_df.at[idx, "Claim_Count"] = len(result["claims"])
        work_df.at[idx, "Claim_Extraction_Raw"] = result["raw_output"]

    except Exception as e:
        print(f"Row {idx} failed:", e)
        work_df.at[idx, "Atomic_Claims"] = "[]"
        work_df.at[idx, "Claim_Count"] = 0
        work_df.at[idx, "Claim_Extraction_Raw"] = f"ERROR: {e}"

    # checkpoint
    if (idx + 1) % 10 == 0:
        work_df.to_csv(OUTPUT_CSV, index=False)

work_df.to_csv(OUTPUT_CSV, index=False)

print("Saved:", OUTPUT_CSV)

In [ ]:
display_cols = [
    "Question_ID",
    "Question",
    "Phi3_Answer",
    "Atomic_Claims",
    "Claim_Count"
]

work_df[display_cols].head(10)

## Convert to claim-level format


In [ ]:
claim_rows = []

for _, row in work_df.iterrows():
    try:
        claims = json.loads(row["Atomic_Claims"])
    except Exception:
        claims = []

    for claim_id, claim in enumerate(claims, start=1):
        claim_rows.append({
            "Question_ID": row.get("Question_ID"),
            "Question": row["Question"],
            "Phi3_Answer": row["Phi3_Answer"],
            "Source": row.get("Source", ""),
            "Claim_ID": claim_id,
            "Atomic_Claim": claim
        })

claims_df = pd.DataFrame(claim_rows)

claims_df.to_csv(CLAIM_LEVEL_CSV, index=False)

print("Number of extracted claims:", len(claims_df))
print("Saved:", CLAIM_LEVEL_CSV)

claims_df.head(20)

## Basic sanity checks


In [ ]:
print("Answers:", len(work_df))
print("Total claims:", work_df["Claim_Count"].sum())
print("Average claims per answer:", work_df["Claim_Count"].mean())
print("Answers with 0 claims:", (work_df["Claim_Count"] == 0).sum())

# Answers with unusually many claims
work_df.sort_values("Claim_Count", ascending=False)[
    ["Question_ID", "Claim_Count", "Phi3_Answer", "Atomic_Claims"]
].head(10)

In [ ]:

df = pd.read_csv(OUTPUT_CSV)
df = df[df["Source"].fillna("").str.contains(r"(^|\.)wikipedia\.org", case=False, regex=True)].copy()

print(f"Rows after Wikipedia-only filter: {len(df)}")

df.to_csv("../Data/Phi3_Claims_Wikipedia_Only.csv", index=False)

In [ ]:
df = pd.read_csv(CLAIM_LEVEL_CSV)
df = df[df["Source"].fillna("").str.contains(r"(^|\.)wikipedia\.org", case=False, regex=True)].copy()

print(f"Rows after Wikipedia-only filter: {len(df)}")

df.to_csv("../Data/Phi3_Claims_Level_Wikipedia_Only.csv", index=False)